This notebook refits the selected model configurations on all labeled seasons (2020/21–2025/26) and saves the production models and metadata used for current-season inference.

### 1. Load Models

In [26]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone

MODEL_DIR = Path("../models")

academic_without_history = joblib.load(
    MODEL_DIR / "xgb_without_history.joblib"
)

academic_with_history = joblib.load(
    MODEL_DIR / "xgb_with_history.joblib"
)

In [27]:
# create fresh production models
production_without_history = clone(
    academic_without_history
)

production_with_history = clone(
    academic_with_history
)

### 2. Train on All Labeled Historic Data

In [28]:
df = pd.read_csv("../data/processed/modeling_dataset_v1.csv")

production_df = df[
    df["season"] <= 2025
].copy()

# create log cols (see 01_eda_modeling_dataset.ipynb for reference)

# market value
production_df["log_target_market_value"] = np.log1p(
    df["target_market_value_in_eur"]
)
# previous market value
production_df["log_previous_market_value"] = np.log1p(
    df["previous_market_value_in_eur"]
)

# define target variable
target_variable = "log_target_market_value"

In [29]:
# set features
categorical_features = [
    "position",
    "sub_position",
    "foot",
    "competition_id"
]

# Define numerical features
numeric_features = [
    "season",

    "age_at_valuation",
    "height_in_cm",

    "total_games_played_season",
    "total_minutes_played",

    "total_goals",
    "goals_per_90_min",
    "total_assists",
    "assists_per_90_min",
    "goal_contributions_per_90_min",

    "total_yellow_cards",
    "yellow_cards_per_90_min",
    "total_red_cards",
    "red_cards_per_90_min",

    "total_clean_sheets",
    "clean_sheet_rate",

    "clubs_played_for_season",
    "club_goals_scored",
    "club_goals_conceded",
    "club_games_played",
    "avg_points_per_game",
    
    "total_previous_transfers",
    "transfers_during_season",
    "has_previous_transfer",
    "days_since_last_transfer"
]

# Define historical features
historical_features = [
    "log_previous_market_value",
    "days_since_previous_valuation"
]


# Join the two lists of features, first without historical market value information
features_without_history = (    
    categorical_features
    + numeric_features
)

# Join the all the features, now with historical information included
features_with_history = (
    features_without_history +
    historical_features
)

In [30]:
X_production_without_history = production_df[
    features_without_history
]

X_production_with_history = production_df[
    features_with_history
]

y_production = production_df[
    target_variable
]

### 3. Fit the Production Models

In [31]:
production_without_history.fit(
    X_production_without_history,
    y_production
)

production_with_history.fit(
    X_production_with_history,
    y_production
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### 4. Save the Models

In [32]:
joblib.dump(
    production_without_history,
    MODEL_DIR / "xgb_without_history_production.joblib"
)

joblib.dump(
    production_with_history,
    MODEL_DIR / "xgb_with_history_production.joblib"
)

['..\\models\\xgb_with_history_production.joblib']

In [33]:
# store production metadata

import json

production_metadata = {
    "model_version": "v1",
    "training_seasons": "2020-2025",
    "target": "log_target_market_value",
    "target_inverse_transform": "expm1",

    "routing": {
        "with_history": "Use when previous market valuation is available",
        "without_history": "Fallback when previous market valuation is unavailable"
    },

    "features_without_history": features_without_history,
    "features_with_history": features_with_history,

    "without_history_params": {
        "objective": "reg:squarederror",
        "n_estimators": 600,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "random_state": 42
    },

    "with_history_params": {
        "objective": "reg:squarederror",
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "random_state": 42
    }
}

with open(
    MODEL_DIR / "production_metadata.json",
    "w"
) as f:
    json.dump(
        production_metadata,
        f,
        indent=4
    )